# 05 — Two-node B60 cluster: aggregate throughput & cost (Ray)

Notebook 04 characterized each B60 desktop on its own. This notebook drives **both at
once** and turns the per-node numbers into the headline cost figures — **tokens/s/$** and
**tokens/s/W** — versus a single RTX 6000 Ada.

**Topology:** vLLM runs the models (one engine per B60, one per node, launched by you and
pinned to its card), and **Ray runs the experiment**. Ray places one load driver on each node so
requests originate locally (no cross-network client skew), starts them together so both cards are
busy at the same time, and collects each node's results centrally to sum into system throughput.
Ray never touches the GPU — it can't see Intel XPUs as resources, and here it doesn't need to.

### Prerequisites

1. **One vLLM engine per node on `:8000`**, each pinned to its local B60 — the exact command you
   validated in notebook 04 (`vllm serve … --max-num-seqs <KV ceiling>`), running on *both* desktops.
2. **A Ray cluster spanning both desktops**, each started with the custom `xpu` resource. From the
   README:
   ```bash
   # desktop1 (head — the box with more RAM):
   ray start --head --resources='{"xpu": 1}'
   # desktop2 (worker): use the address the head printed
   ray start --address='<head-ip>:6379' --resources='{"xpu": 1}'
   ```
   Same Ray + Python version on both, same LAN, GCS port (6379) reachable.
3. Run this notebook **on the head node**.

In [ ]:
import sys, pathlib, json, time
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import ray
from bench import cluster, plotting
import matplotlib.pyplot as plt

REPO_ROOT = pathlib.Path.cwd().parent

# Connect to the already-running cluster. working_dir ships this repo to every
# worker so the driver tasks import the same bench/ code and data/prompts.json.
ray.init(address='auto', runtime_env={'working_dir': str(REPO_ROOT),
                                      'excludes': ['data/**', 'notebooks/**', '.git/**', '.venv/**']})
print('ray', ray.__version__)
for n in ray.nodes():
    if n['Alive']:
        print(f"  node {n['NodeManagerAddress']:<16} xpu={n['Resources'].get('xpu', 0)} "
              f"cpu={n['Resources'].get('CPU', 0)}")
print('total xpu units (= B60 nodes):', cluster.cluster_xpu_count())

## Edit per run

`CONCURRENCY_LEVELS` is **per node** (each driver sweeps these against its local engine), so the
system in-flight count is `level × n_nodes`. Keep `MODEL`, `MAX_TOKENS`, and the levels identical to
your notebook-04 runs so the per-node numbers line up. Cap the levels at each engine's KV ceiling
(notebook 04's `--max-num-seqs`) — sweeping past it just queues.

The **cost constants** are where you set the comparison. The Ada row should use a number you
*measured* the same way (run notebook 04 against the Ada at the same model + context) — a spec
sheet figure isn't apples-to-apples. Prices/watts are per single card; the B60 row is scaled by
`N_B60` inside `cost_metrics`.

In [ ]:
# ---- EDIT PER RUN ----
MODEL              = 'cyankiwi/Qwen3.6-27B-AWQ-INT4'   # MUST match what both engines serve
MAX_TOKENS         = 256
CONCURRENCY_LEVELS = [1, 2, 4, 8]      # PER NODE; keep <= each engine's --max-num-seqs
WINDOW_S           = 60.0              # measured seconds per level
COOLDOWN_S         = 10.0
BASE_URL           = 'http://127.0.0.1:8000/v1'   # evaluated ON each node -> its local engine

# Cost model (per single card). Fill in your real numbers.
N_B60          = 2
B60_PRICE_USD  = 500.0     # one Arc Pro B60
B60_BOARD_W    = 200.0     # one B60 board power under load (measure if you can)
ADA_PRICE_USD  = 7000.0    # one RTX 6000 Ada
ADA_BOARD_W    = 300.0     # one Ada board power
ADA_TOKENS_PER_S = 0.0     # <-- measured Ada aggregate at the SAME model+context (0 = skip cost panel)
# ----------------------

PROMPTS   = json.loads((REPO_ROOT / 'data' / 'prompts.json').read_text())['text_only']
TIMESTAMP = time.strftime('%Y%m%d-%H%M%S')
OUT_PATH  = REPO_ROOT / 'data' / 'b60' / f'cluster_{TIMESTAMP}.json'
assert cluster.cluster_xpu_count() >= 1, "no xpu nodes — start ray with --resources='{\"xpu\":1}'"
print(f"will sweep {CONCURRENCY_LEVELS} req/node x {cluster.cluster_xpu_count()} node(s), "
      f"{WINDOW_S:.0f}s/level, model={MODEL}")

## Run the synchronized cluster sweep

Launches one driver per B60, all at once. Each sweeps its local engine through `CONCURRENCY_LEVELS`
and reports back. Total wall-clock ≈ `len(levels) × (WINDOW_S + COOLDOWN_S)` — the drivers run in
parallel, so adding nodes doesn't add time.

In [ ]:
result = cluster.cluster_sweep(
    model=MODEL, concurrency_levels=CONCURRENCY_LEVELS, window_s=WINDOW_S,
    max_tokens=MAX_TOKENS, prompts=PROMPTS, base_url=BASE_URL, cooldown_s=COOLDOWN_S,
)
agg = cluster.aggregate(result)

print('per-node hosts:', [n['host'] for n in result['nodes']])
print()
print(f"{'req/node':>9} {'total':>6} {'system tok/s':>13}   per-node")
for r in agg:
    per = '  '.join(f"{h}={v:.0f}" for h, v in r['per_node_tokens_per_s'].items())
    print(f"{r['concurrency_per_node']:>9} {r['total_in_flight']:>6} "
          f"{r['total_tokens_per_s']:>13.1f}   {per}")
# any node errors?
for n in result['nodes']:
    errs = sum(l['errors'] for l in n['levels'])
    if errs:
        print(f"  NOTE: {n['host']} logged {errs} request errors")

## Combined throughput

Stacked by node, so an asymmetric desktop (recall desktop2 ran ~18% behind desktop1 — host CPU/RAM,
not the card) shows up as a shorter segment rather than being hidden in an average. The system
capacity is where the total bar stops growing.

In [ ]:
ax = plotting.cluster_throughput(agg)
plt.tight_layout(); plt.show()

## Cost-effectiveness vs a single RTX 6000 Ada

Picks the system's best (peak) combined throughput and compares tokens/s/$ and tokens/s/W against
one Ada. Set `ADA_TOKENS_PER_S` above (measured the same way) to draw this; the B60 usually wins
per-dollar by a wide margin, while per-watt is the closer, more honest fight.

In [ ]:
peak = max(r['total_tokens_per_s'] for r in agg)
print(f"system peak combined throughput: {peak:.1f} tok/s")

cost = None
if ADA_TOKENS_PER_S > 0:
    cost = cluster.cost_metrics(
        total_tokens_per_s=peak, n_b60=N_B60,
        b60_price_usd=B60_PRICE_USD, b60_board_w=B60_BOARD_W,
        ada_tokens_per_s=ADA_TOKENS_PER_S, ada_price_usd=ADA_PRICE_USD, ada_board_w=ADA_BOARD_W,
    )
    for c in cost:
        print(f"  {c['label']:<18} {c['tokens_per_s']:>6.0f} tok/s  "
              f"${c['price_usd']:>6.0f}  {c['board_w']:>4.0f} W   "
              f"{c['tokens_per_s_per_usd']:.3g} tok/s/$   {c['tokens_per_s_per_w']:.3g} tok/s/W")
    fig, _ = plotting.cost_comparison(cost)
    plt.show()
else:
    print('ADA_TOKENS_PER_S = 0 -> cost panel skipped. Measure the Ada with notebook 04 '
          '(same model + context) and set it above.')

In [ ]:
out = cluster.save_cluster_run(result, agg, OUT_PATH, timestamp=TIMESTAMP, cost=cost)
print('saved', out)

## What this gives you (and what to watch)

- **System capacity** = the top of the combined-throughput bars (≈ sum of each node's notebook-04
  knee, since the nodes are independent — separate cards and hosts, no shared endpoint).
- **Don't average the nodes.** Report the real per-node split; a slow node (host CPU/RAM, thermals)
  drags the *aggregate*, and the stacked chart keeps that visible. If you fix desktop2's RAM/clock,
  re-run and the gap should close.
- **Latency under load** is per node — read `worst_ttft_p95` in the aggregate rows; the system is
  only as responsive as its slowest replica. Keep each node at/under its KV ceiling.
- **Ray's role here is orchestration only** — synchronized launch + central aggregation + the
  dashboard (`http://<head>:8265`). The serving and load-balancing are vLLM's / your own; there is
  no single OpenAI endpoint in this topology (that's Option A, Ray Serve).

### Caveats for the write-up

- **Ray ≠ GPU-aware on XPU.** The `xpu` resource is a label for *placement*, not a real device
  count; vLLM does the actual card binding. State the Ray and vLLM versions.
- **Apples-to-apples.** The Ada row must be measured at the same model, quantization, context, and
  `max_tokens`. A spec-sheet tok/s is not a fair comparison.
- **Power.** tokens/s/W is only as good as your watt numbers — prefer a measured board/outlet figure
  over TDP. It's the metric where the Ada is most competitive, so measure it honestly.